In [13]:
!pip install -q playwright img2pdf requests
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libasound2t64 set to manually installed.
libcairo2 is already the newest versi

In [15]:
import asyncio
from playwright.async_api import async_playwright

async def veri_scribd(url: str):
  async with async_playwright() as p:

    browser = await p.chromium.launch(
        headless=True,
        args=['--no-sandbox', '--disable-setuid-sandbox']
    )
    context = await browser.new_context(
        viewport={"width": 1280, "height": 960},
        user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    page = await context.new_page()

    print("[+] Sayfaya gidiliyor...")
    await page.goto(url, wait_until="domcontentloaded")
    await asyncio.sleep(3)

    page_containers = await page.query_selector_all(".page-container, .document_page, [id^='outer_page_']")
    print(f"[+] Tespit edilen sayfa alanı sayısı: {len(page_containers)}")

    await browser.close()

scribd_url = "https://www.scribd.com/document/830951086/%C4%B0%C5%9E-VE-SOSYAL-GUVENL%C4%B0K-HUKUKU-K%C4%B0TAP"
await veri_scribd(scribd_url)

[+] Sayfaya gidiliyor...
[+] Tespit edilen sayfa alanı sayısı: 276


In [18]:
import os
import asyncio
import img2pdf
from playwright.async_api import async_playwright

async def veri_download(url: str, output_pdf: str = "Is_ve_Sosyal_Guvenlik_Hukuku.pdf"):
  output_dir = "scribd_pages"
  os.makedirs(output_dir, exist_ok=True)

  async with async_playwright() as p:
    browser = await p.chromium.launch(
        headless=True,
        args=['--no-sandbox', '--disable-setuid-sandbox']
    )
    context = await browser.new_context(
        viewport={"width": 1400, "height": 1800},
        device_scale_factor=2,
    )
    page = await context.new_page()

    print("[+] Dokuman yukleniyor...")
    await page.goto(url, wait_until="domcontentloaded")
    await asyncio.sleep(4)

    await page.add_style_tag(content="""
      .banner_ad, .ad_wrapper, .expended_doc_promo, .scribd_doc_promo,
      .mobile_promotion, .site_header { display: none !important; }
    """)


    containers = await page.query_selector_all(".page-container, .document_page, [id^='outer_page_']")
    total_pages = len(containers)
    print(f"[+] Toplam {total_pages} sayfa icin indirme baslatiliyor..")
    image_paths = []

    for idx, container in enumerate(containers, start=1):
      await container.scroll_into_view_if_needed()
      await asyncio.sleep(0.7)

      img_path = os.path.join(output_dir, f"page_{idx:03d}.png")

      await container.screenshot(path=img_path)
      image_paths.append(img_path)

      if idx % 20 == 0 or idx == total_pages:
        print(f"[>] Progress: {idx}/{total_pages} pages downloaded...")

  await browser.close()

  print("[+] Gorseller PDF formatinda birlestiriliyor..")
  with open(output_pdf, "wb") as f:
    f.write(img2pdf.convert(image_paths))
  print(f"[TIK] Islem tamam! Olusturulan Dosya: {output_pdf}")

scribd_url = "https://www.scribd.com/document/example"
await veri_download(scribd_url)

[+] Dokuman yukleniyor...
[+] Toplam 276 sayfa icin indirme baslatiliyor..
[>] Progress: 20/276 pages downloaded...
[>] Progress: 40/276 pages downloaded...
[>] Progress: 60/276 pages downloaded...
[>] Progress: 80/276 pages downloaded...
[>] Progress: 100/276 pages downloaded...
[>] Progress: 120/276 pages downloaded...
[>] Progress: 140/276 pages downloaded...
[>] Progress: 160/276 pages downloaded...
[>] Progress: 180/276 pages downloaded...
[>] Progress: 200/276 pages downloaded...
[>] Progress: 220/276 pages downloaded...
[>] Progress: 240/276 pages downloaded...
[>] Progress: 260/276 pages downloaded...
[>] Progress: 276/276 pages downloaded...
[+] Gorseller PDF formatinda birlestiriliyor..
[TIK] Islem tamam! Olusturulan Dosya: Is_ve_Sosyal_Guvenlik_Hukuku.pdf


In [20]:
from google.colab import files
files.download("Is_ve_Sosyal_Guvenlik_HUkuku.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>